In [1]:
import xarray as xr 
from anemoi.datasets import open_dataset
import numpy as np
import yaml
import os 
import zarr
import matplotlib.ticker as mticker
import pandas as pd
import cfgrib
from pathlib import Path
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from collections import defaultdict
import random
import matplotlib.tri as mtri
import matplotlib.ticker as mticker
from random import randint
from earthkit.meteo.thermo.array import (
    specific_humidity_from_relative_humidity as q_from_r_earthkit
)

In [2]:
import anemoi.datasets
from datetime import datetime

# Path to your combined or primary dataset
ds = anemoi.datasets.open_dataset("/mnt/weatherloss/WindPower/data/EGU26/Anemoidatasets/Cerra_A.zarr")

target_date = datetime.fromisoformat("2025-04-15T06:00:00")
try:
    index = ds.datetime_to_index(target_date)
    print(f"Date found at index: {index}")
    print(f"Latitudes available: {len(ds.latitudes)}")
except Exception as e:
    print(f"Error: {e}")
    print(f"Dataset range: {ds.dates[0]} to {ds.dates[-1]}")

Error: 'Zarr' object has no attribute 'datetime_to_index'
Dataset range: 2015-01-01T00:00:00 to 2025-07-31T21:00:00


## Make CERRA & ERA5 zarr

In [22]:
ds=xr.open_dataset("/mnt/weatherloss/WindPower/data/EGU26/cerra_EGU_xy.zarr")

In [23]:
ds

<xarray.Dataset> Size: 180GB
Dimensions:    (y: 196, x: 154, time: 28736, level: 9)
Coordinates:
  * time       (time) datetime64[ns] 230kB 2015-01-01 ... 2024-10-31T21:00:00
  * level      (level) float64 72B 1e+03 950.0 900.0 850.0 ... 700.0 600.0 500.0
    latitude   (y, x) float64 241kB ...
    longitude  (y, x) float64 241kB ...
Dimensions without coordinates: y, x
Data variables: (12/14)
    lsm        (y, x) float32 121kB ...
    mcc        (time, y, x) float32 3GB ...
    msl        (time, y, x) float32 3GB ...
    orog       (y, x) float32 121kB ...
    r          (time, level, y, x) float32 31GB ...
    si10       (time, y, x) float32 3GB ...
    ...         ...
    u          (time, level, y, x) float32 31GB ...
    v          (time, level, y, x) float32 31GB ...
    wdir10     (time, y, x) float32 3GB ...
    wdir100    (time, y, x) float32 3GB ...
    ws100      (time, y, x) float32 3GB ...
    z          (time, level, y, x) float32 31GB ...

In [10]:
import pandas as pd
import xarray as xr

ts = pd.Timestamp("2022-01-03 06:00:00")

# Select time slice
power_t = ds.power.sel(time=ts)

# Build mask for strictly positive power
mask = power_t > 0

# Apply mask to all relevant variables
subset = xr.Dataset({
    "power": power_t.where(mask),
    "turbinecount": ds.turbinecount.where(mask),
    "capacity": ds.capacity.where(mask),
})

# Convert to dataframe and drop empty rows
df = (
    subset
    .to_dataframe()
    .dropna(subset=["power"])
    .reset_index()
    .sort_values("power", ascending=False)
)

print(df)


      y    x   latitude  longitude                time        power  \
18  111  112  53.900816   1.795538 2022-01-03 06:00:00  1123.316650   
22  123   47  54.104898  -3.743323 2022-01-03 06:00:00   643.999695   
7    66  105  51.656612   1.536747 2022-01-03 06:00:00   604.280029   
13   99   99  53.253445   0.808297 2022-01-03 06:00:00   563.806030   
15  110   47  53.471132  -3.574729 2022-01-03 06:00:00   498.753326   
9    68  101  51.737597   1.204824 2022-01-03 06:00:00   480.100006   
4    64  122  51.622934   2.899332 2022-01-03 06:00:00   453.666656   
10   71  110  51.923657   1.901441 2022-01-03 06:00:00   453.580322   
3    63  123  51.576947   2.984126 2022-01-03 06:00:00   441.666656   
0    48   80  50.649069  -0.284997 2022-01-03 06:00:00   391.457336   
12   98  106  53.236120   1.391084 2022-01-03 06:00:00   377.227997   
19  120   50  53.981532  -3.455356 2022-01-03 06:00:00   377.202667   
2    62  123  51.527615   2.989452 2022-01-03 06:00:00   349.000000   
21  12

In [17]:
ds

<xarray.Dataset> Size: 5GB
Dimensions:    (y: 196, x: 154, time: 720, level: 9)
Coordinates:
  * time       (time) datetime64[ns] 6kB 2015-01-01 ... 2015-03-31T21:00:00
  * level      (level) float64 72B 1e+03 950.0 900.0 850.0 ... 700.0 600.0 500.0
    latitude   (y, x) float64 241kB ...
    longitude  (y, x) float64 241kB ...
Dimensions without coordinates: y, x
Data variables: (12/14)
    lsm        (y, x) float32 121kB ...
    mcc        (time, y, x) float32 87MB ...
    msl        (time, y, x) float32 87MB ...
    orog       (y, x) float32 121kB ...
    r          (time, level, y, x) float32 782MB ...
    si10       (time, y, x) float32 87MB ...
    ...         ...
    u          (time, level, y, x) float32 782MB ...
    v          (time, level, y, x) float32 782MB ...
    wdir10     (time, y, x) float32 87MB ...
    wdir100    (time, y, x) float32 87MB ...
    ws100      (time, y, x) float32 87MB ...
    z          (time, level, y, x) float32 782MB ...

In [18]:
ds=xr.open_dataset("/mnt/weatherloss/WindPower/data/EGU26/era5_EGU.zarr")

In [19]:
ds

<xarray.Dataset> Size: 63GB
Dimensions:       (latitude: 85, longitude: 93, time: 31408, level: 9)
Coordinates:
  * latitude      (latitude) float64 680B 63.5 63.25 63.0 ... 43.0 42.75 42.5
  * longitude     (longitude) float64 744B -12.0 -11.75 -11.5 ... 10.75 11.0
  * time          (time) datetime64[ns] 251kB 2015-01-01 ... 2025-09-30T21:00:00
  * level         (level) float64 72B 1e+03 950.0 900.0 ... 700.0 600.0 500.0
Data variables: (12/20)
    capacity      (latitude, longitude) float32 32kB ...
    lsm           (latitude, longitude) float32 32kB ...
    mcc           (time, latitude, longitude) float32 993MB ...
    msl           (time, latitude, longitude) float32 993MB ...
    power         (time, latitude, longitude) float32 993MB ...
    q             (time, level, latitude, longitude) float32 9GB ...
    ...            ...
    u100          (time, latitude, longitude) float32 993MB ...
    v             (time, level, latitude, longitude) float32 9GB ...
    v10           (time, latitude, longitude) float32 993MB ...
    v100          (time, latitude, longitude) float32 993MB ...
    z             (time, level, latitude, longitude) float32 9GB ...
    z_sfc         (latitude, longitude) float32 32kB ...

In [34]:
import xarray as xr
import numpy as np

ds = xr.open_zarr("/mnt/weatherloss/WindPower/data/EGU26/era5_EGU.zarr")

p = ds["power"]
has_any = (~p.isnull()).any(("latitude", "longitude"))  # bool over time
idx = int(np.argmax(has_any.values))  # first True (if any True exists)

print("Any valid power at all?", bool(has_any.any().values))
print("First time with any valid power:", p.time.values[idx])

Any valid power at all? True
First time with any valid power: 2020-01-01T00:00:00.000000000


In [35]:
has_any = (~ds["power"].isnull()).any(("latitude","longitude"))
per_year = has_any.groupby("time.year").sum()
print(per_year.to_series())

year
2015       0
2016       0
2017       0
2018       0
2019       0
2020    2928
2021    2920
2022    2920
2023    2920
2024    2928
2025    1696
Name: power, dtype: int64


In [36]:
ds = xr.open_zarr("/mnt/weatherloss/WindPower/data/EGU26/cerra_EGU.zarr")


FileNotFoundError: No such file or directory: '/mnt/weatherloss/WindPower/data/EGU26/cerra_EGU.zarr'

In [40]:
ds = xr.open_zarr("/mnt/weatherloss/WindPower/data/EGU26/cerra_EGU_xy.zarr")

for v in ["capacity", "turbinecount", "turbinemask"]:
    da = ds[v]
    print(v)
    print("min:", float(da.min().compute()))
    print("max:", float(da.max().compute()))
    print("mean:", float(da.mean().compute()))
    print()

capacity
min: 0.0
max: 2604.0
mean: 0.4258282482624054

turbinecount
min: 0.0
max: 339.0
mean: 0.08057248592376709

turbinemask
min: 1.0
max: 1.0
mean: 1.0



In [41]:
ds = xr.open_zarr("/mnt/weatherloss/WindPower/data/EGU26/era5_EGU.zarr")

for v in ["capacity", "turbinecount", "turbinemask"]:
    da = ds[v]
    print(v)
    print("min:", float(da.min().compute()))
    print("max:", float(da.max().compute()))
    print("mean:", float(da.mean().compute()))
    print()

capacity
min: 90.0
max: 2604.0
mean: 676.4841918945312

turbinecount
min: 30.0
max: 339.0
mean: 128.0

turbinemask
min: 1.0
max: 1.0
mean: 1.0

